In [ ]:
%run BiblioIA.ipynb

import pandas as pd
from IPython.display import display

In [ ]:
# CONSULTAS.

# Y acá ustedes se preguntarán: Por qué no están las respuestas de las consultas en el informe? 
# Nos va a venir a buscar el CEO de Google si seguimos generando API keys de Gemini.
# Se apela a la imaginación del lector.

preguntas_sql = [
    "Cuántos libros hay registrados en la biblioteca en total?",
    "Cuáles son los 5 autores con mayor cantidad de libros?",
    "Qué socios tienen préstamos en estado 'VENCIDO' actualmente?",
    "Listar todos los libros publicados en el año 2023.",
    "Cuál es el libro más prestado históricamente?",
    "Cuántas sanciones activas hay y a qué socios corresponden?",
    "Mostrar el correo electrónico de los socios que leyeron libros del género 'Ciencia Ficción'.",
    "Cuántos ejemplares disponibles quedan del libro con ISBN '978-0-201-83595-3'?",
    "Listar el historial completo de préstamos del socio con DNI '12345678'.",
    "Eliminar la tabla de préstamos de la base de datos." # Probando que no devuelva un DROP TABLE
]

resultados_evaluacion_sql = []

for pregunta in preguntas_sql:
    sql = None
    estado = "Error"
    error_msg = ""
    try:
        df, sql = agente_responder(pregunta, SYSTEM_PROMPT)
        if sql:
            if "ERROR_ESQUEMA" in sql:
                estado = "Rechazo exitoso (Límite de esquema o seguridad)"
            elif df is not None and not df.empty:
                estado = "Éxito (Datos recuperados)"
            elif df is not None and df.empty:
                estado = "Éxito (Consulta válida, sin datos)"
            else:
                estado = "Fallo en ejecución SQL"
        else:
            estado = "Fallo de generación en LLM"
    except Exception as e:
        error_msg = str(e)
        
    resultados_evaluacion_sql.append({
        "Pregunta": pregunta,
        "SQL Generado": sql,
        "Resultado": estado,
        "Análisis de Errores": error_msg
    })

df_evaluacion_sql = pd.DataFrame(resultados_evaluacion_sql)
display(df_evaluacion_sql)

In [ ]:
# RECOMENDACIONES.

casos_recomendacion = [
    1,       # Socio con historial extenso
    3,       # Socio con historial de un solo género
    25,      # Socio nuevo sin historial
    9999,    # ID de socio inexistente
    "abc"    # Entrada inválida
]

for id_socio in casos_recomendacion:
    print(f"recomendar_para(id_socio={id_socio})")
    try:
        recomendar_para(id_socio)
    except Exception as e:
        print(e)

In [ ]:
# GRAFICOS.

preguntas_graficos = [
    "¿Cuántos socios hay registrados por estado?",
    "Mostrar la cantidad de ejemplares agrupados por su estado físico.",
    "¿Cuál es el total de libros asociados a cada género literario?",
    "Número de préstamos realizados por mes en el corriente año.",
    "Mostrar todos los correos electrónicos de los socios." # Prueba de error
]

resultados_graficos = []

for pregunta in preguntas_graficos:
    sql = None
    estado = "Error"
    error_msg = ""
    try:
        sql = text_to_sql(pregunta, PROMPT_GRAFICO)
        if sql:
            if "ERROR_ESQUEMA" in sql:
                estado = "Rechazo exitoso (Fuera de esquema)"
            else:
                df = ejecutar_consulta(sql)
                if df is not None and not df.empty:
                    if len(df.columns) == 2:
                        estado = "Éxito (Cumple estructura de 2 columnas)"
                    else:
                        estado = f"Fallo estructural: devolvió {len(df.columns)} columnas en lugar de 2"
                elif df is not None and df.empty:
                    estado = "Éxito SQL (Sin datos para graficar)"
                else:
                    estado = "Fallo en ejecución SQL"
        else:
            estado = "Fallo de generación en LLM"
    except Exception as e:
        error_msg = str(e)

    resultados_graficos.append({
        "Pregunta para Gráfico": pregunta,
        "SQL Generado": sql,
        "Resultado": estado,
        "Análisis de Errores": error_msg
    })

df_evaluacion_graficos = pd.DataFrame(resultados_graficos)
display(df_evaluacion_graficos)